# Task 3: General Questions

This notebook covers four general questions that apply across both datasets:

1. Data quality issues and how to handle them
2. Outliers: what exists in the data and how to treat them
3. Key takeaways for a non-technical audience
4. Attribution model proposal for N26

In [2]:
import pandas as pd
import numpy as np
import sys, os
sys.path.append(os.path.abspath(os.path.join('..')))
from src.cleaning import fix_date_column

---

## Q1: Data Quality Issues

### Dataset A (funnel data)

The dataset had 12,164 rows across date, country, and marketing channel. Three issues came up:

**Impossible dates.** 70 rows contained dates that do not exist: February 29 in non-leap years (2017, 2018, 2019) and February 30. These were not fixable by inference since there is no way to know the intended date. I dropped them. 12,094 rows remained.

**Duplicate rows.** A small number of rows shared the same date, country, and channel combination. I kept the first occurrence and dropped the rest. The deduplication logic is in `src/cleaning.py` and uses a row number window function.

**No missing values.** After date parsing and deduplication, all five funnel columns (signups through first_transaction) were complete with no nulls.

### Dataset B (experiment data)

This dataset was clean. All 5,000 rows had valid dates within the March 2017 window, no missing values across any column, and no duplicate user IDs. The date column required parsing from string to datetime, which `fix_date_column` handled without dropping any rows.

### General handling approach

How I handle data quality issues depends on the type:

| Issue | Approach | Rationale |
|-------|----------|----------|
| Unparseable / impossible dates | Parse with `errors='coerce'`, then drop NaTs | No safe way to impute a date that should not exist |
| Duplicate rows | Depends. Usually drop if the dataset is big enough. Here i did = Keep first occurrence, document the strategy | Summing duplicates would inflate counts; dropping all would lose real data |
| Missing values in funnel counts | Depends on missingness mechanism. If missing at random and volume is small, drop. If systematic (e.g. a channel not active on certain dates), fill with 0 or model | Imputing funnel counts with means would distort conversion rates |
| Missing experiment outcomes | Treat as no conversion if the user was confirmed exposed. If exposure itself is uncertain, exclude the user | Experiment integrity depends on a clean exposure log |

One thing worth flagging: the funnel data has extremely tight numeric ranges (signups go from 90 to 100, std of ~3). That level of uniformity is not realistic for production data. In a real dataset I would also check for truncation or rounding artifacts, and verify that counts at each funnel step cannot exceed the step above.

In [3]:
df_a = pd.read_csv("../data/part_a_dataset.csv")
df_b = pd.read_csv("../data/part_b_dataset.csv")

print("=== Dataset A ===")
print(f"Shape: {df_a.shape}")
print(f"Nulls:\n{df_a.isnull().sum().to_string()}")
print(f"Full duplicates: {df_a.duplicated().sum()}")
print(f"Key duplicates (date/country/channel): {df_a.duplicated(subset=['date','country','marketing_channel']).sum()}")

print("\n=== Dataset B ===")
print(f"Shape: {df_b.shape}")
print(f"Nulls:\n{df_b.isnull().sum().to_string()}")
print(f"Duplicate user IDs: {df_b.duplicated(subset=['userid']).sum()}")

=== Dataset A ===
Shape: (12884, 8)
Nulls:
date                 0
country              0
marketing_channel    0
signups              0
kyc_init             0
kyc_complete         0
card_activation      0
first_transaction    0
Full duplicates: 0
Key duplicates (date/country/channel): 720

=== Dataset B ===
Shape: (5000, 6)
Nulls:
userid                0
country               0
day_exposed_to_ad     0
treatment             0
signed_up_metal       0
signed_up_standard    0
Duplicate user IDs: 0


In [4]:
# Show the impossible dates that were dropped
bad_dates = df_a[pd.to_datetime(df_a['date'], errors='coerce').isna()]
print(f"Rows with unparseable dates: {len(bad_dates)}")
print(bad_dates['date'].value_counts())

Rows with unparseable dates: 74
date
2019-2-29    18
2017-2-29    13
2018-2-29    13
2019-2-30    11
2017-2-30    10
2018-2-30     9
Name: count, dtype: int64


/var/folders/d7/zcztcfjn0nb5hm04571js_g80000gn/T/ipykernel_25635/1819265008.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  bad_dates = df_a[pd.to_datetime(df_a['date'], errors='coerce').isna()]


---

## Q2: Outliers

### Dataset A

At the raw count level (signups, kyc_init, etc.), there are no outliers. The distributions are narrow and symmetric: signups range from 90 to 100 with a std of 3.2. A z-score scan at either sigma=2 or sigma=3 returns nothing.

At the conversion rate level, the picture changes depending on the grain of analysis:

| Sigma threshold | Rows flagged | Pct of total |
|----------------|-------------|-------------|
| z > 2 | 774 / 3,835 | 20.2% |
| z > 3 | 120 / 3,835 | 3.1% |

These flagged rows are not data errors. The median signup volume for flagged rows is 94-98, vs 285-291 for clean rows. The conversion rate variance is higher in thin slices simply because small denominators amplify noise. A segment with 94 signups and 27 KYC completions will show a more extreme rate than one with 862 signups and 240 completions, even if the underlying process is identical.

So there are no meaningful outliers in Dataset A. The flagged rows at sigma=2 are small-sample noise.

### Dataset B

No outliers are possible here. The outcome variables (`signed_up_metal`, `signed_up_standard`) are binary (0 or 1), and the exposure date column is bounded to March 2017. The user-level data is clean.

### How I handle outliers generally

My approach depends on what kind of outlier it is:

**Data errors** (impossible values, system glitches, logging bugs): fix or drop. These are not real observations.

**Thin-sample noise** (like the conversion rate rows above): do not remove. Apply a volume threshold instead. If a segment has fewer than N signups, exclude it from the analysis or flag it separately. The threshold depends on the minimum sample needed to produce a stable rate.

**Genuine extreme values** (real users or days with unusually high/low activity): investigate before touching. A spike in signups on a specific date might be a viral moment or a promo launch, not an error. Removing it would bias the analysis. I would check the business context first, then decide.

**For statistical tests**: if a test assumes normality (ANOVA, t-test), check if the outlier meaningfully affects the result. If it does, use a robust alternative (Welch ANOVA, Mann-Whitney) rather than removing data.

In [5]:
# Dataset A: raw count distributions
df_a_clean = fix_date_column(df_a, date_col='date')
df_a_clean[['signups','kyc_init','kyc_complete','card_activation','first_transaction']].describe().round(2)

[fix_date_column] 74 unparseable date(s):
date
2019-2-29    18
2017-2-29    13
2018-2-29    13
2019-2-30    11
2017-2-30    10
2018-2-30     9
[fix_date_column] dropped 74 row(s); 12810/12884 remain.


,signups,kyc_init,kyc_complete,card_activation,first_transaction
count,12810.00,12810.00,12810.00,12810.00,12810.00
mean,95.02,75.01,54.95,45.06,34.96
std,3.17,3.18,3.17,3.17,3.16
min,90.00,70.00,50.00,40.00,30.00
25%,92.00,72.00,52.00,42.00,32.00
50%,95.00,75.00,55.00,45.00,35.00
75%,98.00,78.00,58.00,48.00,38.00
max,100.00,80.00,60.00,50.00,40.00


In [6]:
# Dataset B: outcome variable distributions
df_b[['signed_up_metal','signed_up_standard']].describe()

,signed_up_metal,signed_up_standard
count,5000.0000,5000.000000
mean,0.0668,0.145800
std,0.2497,0.352941
min,0.0000,0.000000
25%,0.0000,0.000000
50%,0.0000,0.000000
75%,0.0000,0.000000
max,1.0000,1.000000


---

## Q3: Key Takeaways for a Non-Technical Audience

### What we looked at

Two datasets were analysed: three years of acquisition funnel data across 11 marketing channels and 12 countries (Task 1), and a one-month A/B experiment comparing two Facebook ad creatives (Task 2).

---

### Finding 1: The acquisition funnel loses 63% of users before they make their first transaction

Out of every 100 people who sign up, only 37 complete a first transaction. The biggest single drop happens at the identity verification step (KYC): about 27% of users who start the KYC process do not finish it. This is the primary leak in the funnel and the highest-value area to fix.

**What this means:** Improving the KYC completion rate by even a few percentage points would have a larger impact on active customers than increasing signup volume.

---

### Finding 2: No marketing channel converts meaningfully better than another

Across all 11 channels (paid search, organic, social, affiliate, referral, etc.), the end-to-end conversion rate is essentially the same: between 36.4% and 37.1%. This 0.7 percentage point spread is statistically noise, not a real difference.

**What this means:** Channel budget decisions cannot be made on conversion rate alone with this data. To compare channels properly, we need cost data (cost per signup, cost per acquisition) and downstream value data (revenue per user, retention). Volume is the only axis this dataset offers for differentiation.

---

### Finding 3: The Standard card creative outperformed the Metal creative on Facebook

In a March 2017 Facebook experiment, people who saw the Standard card ad signed up for the Standard product at a rate of 15.3%. People who saw the Metal card ad signed up for the Metal product at only 6.7%. That gap is statistically significant and reliable.

A notable side effect: people shown the Metal ad were actually converting to the Standard product at twice the rate they converted to Metal. This could mean the Metal creative generated interest in N26 generally, but not in the Metal tier specifically. Whether this is a creative problem or a product pricing problem (Metal is premium and may be too expensive for this audience) cannot be determined from this data alone.

**What this means:** Scale the Standard creative for this audience. Before redesigning or pausing the Metal creative, run a test with funnel tracking after the click (landing page drop-off, paywall interaction) to understand where the Metal audience is actually lost.

---

### Summary table

| Topic | Finding | Recommended action |
|-------|---------|-------------------|
| Funnel efficiency | 63% of signups never transact; KYC is the biggest drop | Investigate and reduce KYC drop-off |
| Channel performance | All channels convert equally (36-37%) | Add cost and LTV data before shifting budget |
| Facebook A/B test | Standard creative wins clearly on target conversion | Scale Standard; instrument Metal funnel before decisions |
| Metal vs Standard demand | Metal creative audience gravitates to Standard | May reflect product fit gap, not just creative weakness |

---

## Q4: Attribution Model Proposal for N26

### The context

N26 is a mobile bank. The acquisition journey is typically short relative to most financial products: a user sees an ad, researches the product, signs up, and (if they pass KYC) activates a card and transacts. The touchpoint sequence might be: paid social ad > organic search > direct app store download. There is no lengthy CRM nurture sequence or offline touchpoints, which simplifies the attribution problem.

The dataset has 11 distinct marketing channels. The fact that conversion is uniform across channels (Task 1 finding) suggests these channels are likely serving different stages of the funnel rather than independently driving conversions. Paid social and display likely generate awareness; paid search and organic capture intent; direct and app store close the conversion.

### Why last-touch does not work here

Last-touch attribution gives 100% of the credit to the final touchpoint. For N26, this would over-credit direct, organic search, and the app store, while under-crediting the awareness channels that put the brand on the user's radar. It would systematically defund top-of-funnel spending, which is where new customer acquisition for a bank typically starts.

### What I propose

**Short term: position-based (U-shaped) attribution**

Assign 40% of credit to the first touchpoint (what introduced the user to N26), 40% to the last (what converted them), and split the remaining 20% equally across any middle touchpoints.

This is a practical choice for now because:
- It does not require machine learning or large multi-touch datasets to implement
- It acknowledges that both awareness and conversion matter without treating every touchpoint identically
- It stops the systematic under-crediting of channels like paid social and display that rarely appear as the last touch

**Long term: data-driven attribution using Shapley values**

Once multi-touch journey data is available at scale (user ID linked across ad exposures, sessions, and conversions), Shapley value attribution is the right model. It comes from cooperative game theory and assigns credit based on each channel's marginal contribution to the conversion, averaged across all possible orderings of channels in the journey. It makes no assumptions about where in the funnel a channel appears and handles channel interactions properly.

Markov chain models are a close alternative and easier to explain visually (they model the probability that a user converts given a sequence of channels), but Shapley values are more robust to path redundancy.

### One important caveat

Whatever model is used, the conversion event should be defined as **KYC completion or card activation**, not just signup. Signup is a weak signal: 63% of signups never transact. A channel that drives a lot of signups but few KYC completions is not a good channel, even if last-touch or linear attribution makes it look like one. Attribution gated on a meaningful downstream event will reflect actual business value, not just top-of-funnel volume.

### Summary

| Model | Use case | Limitation |
|-------|---------|----------|
| Last-touch | Quick reporting baseline | Over-credits bottom-funnel, defunds awareness |
| Position-based (U-shaped) | Recommended starting point | Weights are arbitrary (40/20/40), not data-derived |
| Shapley values | Recommended long-term model | Requires clean multi-touch journey data at scale |

The conversion event should always be gated on KYC completion or card activation, not signup.